# 7. Dashboard di predizione

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FedeGambe/Master_s_thesis_Data_science/blob/main/notebooks/07_dashboard.ipynb)

Dashboard interattiva (Dash) che usa il modello finale salvato in `modelli/`. In locale si può avviare anche con `python app/dashboard.py`.

In [ ]:
# Configurazione dell'ambiente (locale o Google Colab)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("Master_s_thesis_Data_science"):
        !git clone -q -b main https://github.com/FedeGambe/Master_s_thesis_Data_science.git
    %cd Master_s_thesis_Data_science
    !pip install -q -r requirements-colab.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import plotly.io as pio
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)
if os.environ.get("TESI_GRAFICI_STATICI"):
    pio.renderers.default = "png"  # usato solo per salvare output visibili su GitHub
    pio.defaults.default_width = 1000

Correzioni rispetto alla dashboard originale:

* il genere era codificato al contrario rispetto al training (Maschio = 0 invece di 1)
* gli input non venivano standardizzati prima della trasformazione polinomiale
* veniva mostrata come "accuratezza della predizione" l'accuratezza globale del modello
* il preprocessing era duplicato a mano; ora la dashboard usa la stessa funzione `dati.codifica_ml` del training

In [ ]:
sys.path.insert(0, "app")
from dashboard import crea_app, carica_modello_finale, predici
modello, meta = carica_modello_finale()
print(f"Modello: {meta['modello']} - ROC-AUC test: {meta['metriche_test']['roc_auc']:.3f}")

## Esempio di predizione

In [ ]:
esempio = {"Genere": "Maschio", "Classe d'età": "45-54", "Reddito familiare": 225000,
           "Livello di istruzione": "Laurea 2L o Dottorato", "Casa di proprietà": "Si", "Casa Indipendente": "Si",
           "Numero persone in famiglia": 3, "Numero di auto in famiglia": 2,
           "Importanza di ridurre le emissioni di gas serra": 2.5, "Tipologia di auto precedente": "ICE",
           "Viaggio più lungo negli ultimi 12 mesi": 300, "Numero di viaggi superiori a 200 miglia negli ultimi 12 mesi": 1,
           "Distanza casa-lavoro": 12, "VMT annuo": 11000}
print(f"Probabilità che il veicolo sia un BEV: {predici(modello, esempio):.1%}")

## Avvio della dashboard

In [ ]:
app = crea_app(modello, meta)
if not os.environ.get("TESI_GRAFICI_STATICI"):  # non avviare il server durante la generazione automatica
    app.run(jupyter_mode="inline" if not IN_COLAB else "external", port=8050)